### 문제1

In [ ]:
from ortools.linear_solver import pywraplp
solver = pywraplp.Solver.CreateSolver("SCIP")

# 의사결정 변수
# x = 사우디산 원유 구매량 (배럴)  , y = 베네수엘라산 원유 구매량 (배럴)
# 목적함수
# min ( 60*x + 55*y )   : 원유 구입 비용 (달러)
# 제약식
# 0 <= x <= 9000     : 사우디산 원유 구매량 범위 (배럴)
# 0 <= y <= 6000     : 베네수엘라산 원유 구매량 범위 (배럴)
# 0.3*x + 0.4*y >= 2000    : 가솔린 최소 필요량 (배럴)
# 0.4*x + 0.2*y >= 1500    : 제트유 최소 필요량 (배럴)
# 0.2*x + 0.3*y >= 500     : 윤활유 최소 필요량 (배럴)


# 의사결정 변수   : 각 원유 구입량
x = solver.NumVar(0, 9000, 'x')   # 사우디산 원유 구입량 (단위:배럴)  ... 최대 9000 배럴
y = solver.NumVar(0, 6000, 'y')   # 베네수엘라산 원유 구입량 (단위:배럴)  ... 최대 6000 배럴

# 제약조건
solver.Add(0.3*x + 0.4*y >= 2000)   # 가솔린 
solver.Add(0.4*x + 0.2*y >= 1500)   # 제트유
solver.Add(0.2*x + 0.3*y >= 500)   # 윤활유

# 목적함수
solver.Minimize(60*x + 55*y)   # 최소 구매비용 (단위:$)

status = solver.Solve()

#출력
if status == pywraplp.Solver.OPTIMAL :
    print("OPTIMAL")
    print("목적함수 값 (원유 구입비용) = %.1f ($)" %(solver.Objective().Value()))
    print("사우디산 원유 구입량 (x) = %.1f (배럴)" %(x.solution_value()))
    print("베네수엘라산 원유 구입량 (y) = %.1f (베럴)" %(y.solution_value()))

else :
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수 값 (원유 구입비용) = 312500.0 ($)
사우디산 원유 구입량 (x) = 2000.0 (배럴)
베네수엘라산 원유 구입량 (y) = 3500.0 (베럴)


##### 문제1 nicer version

In [ ]:
from ortools.linear_solver import pywraplp 
solver = pywraplp.Solver.CreateSolver("SCIP")

data = {}
data["constraint_coeffs"] = [[0.3, 0.4] ,    # 각 원유 1배럴 당 가솔린 
                             [0.4, 0.2] ,    # 각 원유 1배럴 당 제트유
                             [0.2, 0.3] ]    # 각 원유 1배럴 당 윤활유 
data["need"] = [2000, 1500, 500]    # 가솔린, 제트유, 윤활유 최소 필요량
data["prices"] = [60, 55]           # 사우디산, 베네수엘라산 원유 구매 가격
data["num_vars"] = 2                # 변수 개수 (원유 2종류)
data["num_constraints"] = 3         # 제약식 개수
data["possible"] = [9000, 6000]     # 각 원유 최대 구매 가능량 

#의사결정 변수
x = {}
for j in range(data["num_vars"]):
    x[j] = solver.NumVar(0, data["possible"][j], "x[%i]" % j)   # 각 원유는 최대 구매가능량 제약 존재 

# 제약식
for i in range(data['num_constraints']):
    constraint_expr = [data['constraint_coeffs'][i][j] * x[j] for j in range(data['num_vars'])] 
    solver.Add(sum(constraint_expr) >= data['need'][i])     # 최소 필요량 이상으로 얻을 수 있게 정제해야함 

# 목적함수
objective = solver.Objective()
obj_expr = [data['prices'][j] * x[j] for j in range(data['num_vars'])] 
solver.Minimize(solver.Sum(obj_expr))

print(f"Solving with {solver.SolverVersion()}")
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("구입비용 = %.1f ($)" %(solver.Objective().Value())) 
    print("( 원유1 = 사우디산 원유 , 원유2 = 베네수엘라산 원유 )")
    for j in range(data["num_vars"]):
        print('원유%d 구입량 = %.1f (배럴) '%(j+1 , x[j].solution_value()))
else :
    print("The problem does not have an optimal solution.")

Solving with SCIP 9.2.2 [LP solver: SoPlex 7.1.3]
구입비용 = 312500.0 ($)
( 원유1 = 사우디산 원유 , 원유2 = 베네수엘라산 원유 )
원유1 구입량 = 2000.0 (배럴) 
원유2 구입량 = 3500.0 (배럴) 


### 문제2

In [ ]:
from ortools.linear_solver import pywraplp
solver = pywraplp.Solver.CreateSolver("SCIP")

# 의사결정 변수
# x = 원석1 구매량 (톤)  , y = 원석2 구매량 (톤)
# 목적함수
# min ( 10*x + 15*y )   : 원석 구매 비용 (만원)
# 제약식
# 0 <= x
# 0 <= y
# 0.002*x + 0.003*y >= 8     : 가돌리늄 최소 필요량 (톤)
# 0.0015*x + 0.0025*y >= 6   : 홀룸 최소 필요량  (톤)
# 0.002*x + 0.001*y >= 4     : 톨륨 최소 필요량  (톤)

# 의사결정 변수   : 각 원석 구입량
x = solver.NumVar(0, solver.infinity(), 'x')   # 원석1 (단위 : 톤)
y = solver.NumVar(0, solver.infinity(), 'y')   # 원석2 (단위 : 톤)

# 제약조건 : 귀금속 추출량 (단위 : 톤)
solver.Add(0.002*x + 0.003*y >= 8)   # 가돌리늄
solver.Add(0.0015*x + 0.0025*y >= 6)   # 홀름
solver.Add(0.002*x + 0.001*y >= 4)   # 툴륨

# 목적함수
solver.Minimize(10*x + 15*y)   # 최소 구매비용 (단위 : 만원)

status = solver.Solve()

#출력
if status == pywraplp.Solver.OPTIMAL :
    print("OPTIMAL")
    print("목적함수 값 (원석 구매비용) = %.1f (만원)" %(solver.Objective().Value()))
    print("원석1 구입량 (x) = %.1f (톤)" %(x.solution_value()))
    print("원석2 구입량 (y) = %.1f (톤)" %(y.solution_value()))

else :
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수 값 (원석 구매비용) = 40000.0 (만원)
원석1 구입량 (x) = 1000.0 (톤)
원석2 구입량 (y) = 2000.0 (톤)


##### 문제2 nicer version

In [ ]:
from ortools.linear_solver import pywraplp 
solver = pywraplp.Solver.CreateSolver("SCIP")

data = {}
data["constraint_coeffs"] = [[0.002 , 0.003] ,     # 각 원석 1톤 당 가돌리늄 함유량
                             [0.0015 , 0.0025] ,   # 각 원석 1톤 당 홀룸 함유량
                             [0.002 , 0.001] ]     # 각 원석 1톤 당 툴륨 함유량
data["need"] = [8, 6, 4]       # 가돌리늄, 홀룸, 툴륨 최소 필요량
data["prices"] = [10, 15]      # 원석1,2 1톤 당 구매가격
data["num_vars"] = 2           # 변수 개수 (원석 2종류)
data["num_constraints"] = 3    # 제약식 개수

#의사결정 변수
infinity = solver.infinity()
x = {}
for j in range(data["num_vars"]):
    x[j] = solver.NumVar(0, infinity, "x[%i]" % j)

# 제약식
for i in range(data['num_constraints']):
    constraint_expr = [data['constraint_coeffs'][i][j] * x[j] for j in range(data['num_vars'])] 
    solver.Add(sum(constraint_expr) >= data['need'][i])   # 최소 필요량 이상으로 얻을 수 있게 채취해야함 

# 목적함수
objective = solver.Objective()
obj_expr = [data['prices'][j] * x[j] for j in range(data['num_vars'])] 
solver.Minimize(solver.Sum(obj_expr))

print(f"Solving with {solver.SolverVersion()}")
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("구입비용 = %.1f (만원)" %(solver.Objective().Value())) 
    for j in range(data["num_vars"]):
        print('원석%d 구입량 = %.1f (톤) '%(j+1 , x[j].solution_value()))
else :
    print("The problem does not have an optimal solution.")

Solving with SCIP 9.2.2 [LP solver: SoPlex 7.1.3]
구입비용 = 40000.0 (만원)
원석1 구입량 = 1000.0 (톤) 
원석2 구입량 = 2000.0 (톤) 


### 문제3

In [5]:
from ortools.linear_solver import pywraplp
solver = pywraplp.Solver.CreateSolver("SCIP")

# 의사결정 변수
# x = 유모차 생산량 (대)  , y = 보행기 생산량 (대)  , z = 자전거 생산량 (대)
# 목적함수
# max ( 30*x + 20*y + 16*z + 3.5*{240-(8*x + 3*y + 3*z)} + 3*{100-(4*x + 2*z)} )   : 최대 이익 (만원)
# 제약식
# 0 <= x
# 0 <= y
# 0 <= z
# 8*x + 3*y + 3*z <= 240     : 기계1 최대 활용가능시간 (시간)
# 4*x + 4*y <= 200   : 기계2 최대 활용가능시간 (시간)
# 4*x + 2*z <= 100     : 기계3 최대 활용가능시간 (시간)
# 기계1 대여가능시간 = { 240 - (8*x + 3*y + 3*z) }
# 기계3 대여가능시간 = { 100 - (4*x + 2*z) }

# 의사결정 변수   :  각 제품 생산량
x = solver.IntVar(0, solver.infinity(), 'x')   # 유모차 (단위 : 대)
y = solver.IntVar(0, solver.infinity(), 'y')   # 보행기 (단위 : 대)
z = solver.IntVar(0, solver.infinity(), 'z')   # 자전거 (단위 : 대)

# 제약조건  :  기계 사용시간 
solver.Add(8*x + 3*y + 3*z <= 240)   # 기계1 (단위:시간)
solver.Add(4*x + 4*y <= 200)   # 기계2 (단위:시간)
solver.Add(4*x + 2*z <= 100)   # 기계3 (단위:시간)

# 목적함수  :  판매이익 + 기계1 대여이익 + 기계3 대여이익
solver.Maximize(30*x + 20*y + 16*z + 3.5*(240-(8*x + 3*y + 3*z)) + 3*(100-(4*x + 2*z)))   # 최대 이익  (단위:만원)

status = solver.Solve()

#출력
if status == pywraplp.Solver.OPTIMAL :
    print("OPTIMAL")
    print("목적함수 값 (총 이익) = %.1f (만원)" %(solver.Objective().Value()))
    print("유모차 생산량 (x) = %.1f (대)" %(x.solution_value()))
    print("보행기 생산량 (y) = %.1f (대)" %(y.solution_value()))
    print("자전거 생산량 (z) = %.1f (대)" %(z.solution_value()))
    print("기계1 대여시간 = %.1f (시간)" %(240-(8*x.solution_value() + 3*y.solution_value() + 3*z.solution_value())))
    print("기계3 대여시간 = %.1f (시간)" %(100-(4*x.solution_value() + 2*z.solution_value())))

else :
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수 값 (총 이익) = 1615.0 (만원)
유모차 생산량 (x) = 0.0 (대)
보행기 생산량 (y) = 50.0 (대)
자전거 생산량 (z) = 0.0 (대)
기계1 대여시간 = 90.0 (시간)
기계3 대여시간 = 100.0 (시간)


##### 문제3 nicer version

In [30]:
from ortools.linear_solver import pywraplp 
solver = pywraplp.Solver.CreateSolver("SCIP")

data = {}
data["constraint_coeffs"] = [[8, 3, 3] ,     # 각 제품당 1개 생산할 때, 기계1 사용시간
                             [4, 4, 0] ,     # 각 제품당 1개 생산할 때, 기계2 사용시간
                             [4, 0, 2] ]     # 각 제품당 1개 생산할 때, 기계3 사용시간
data["supply"] = [240, 200, 100]     # 각 기계 최대 활용가능시간
data["prices"] = [30, 20, 16]        # 각 제품 판매가격
data["num_vars"] = 3                 # 변수 개수 (제품 3종류)
data["num_constraints"] = 3          # 제약식 개수 

#의사결정 변수
infinity = solver.infinity()
x = {}
for j in range(data["num_vars"]):
    x[j] = solver.NumVar(0, infinity, "x[%i]" % j)

# 제약식
for i in range(data['num_constraints']):
    constraint_expr = [data['constraint_coeffs'][i][j] * x[j] for j in range(data['num_vars'])] 
    solver.Add(sum(constraint_expr) <= data['supply'][i])    # 최대 활용가능시간 이하로 기계 사용

# 목적함수
objective = solver.Objective()
obj_expr = [data['prices'][j] * x[j] for j in range(data['num_vars'])] 

obj = solver.Sum(obj_expr) + 3.5*(240-(8*x[0] + 3*x[1] + 3*x[2])) + 3*(100-(4*x[0] + 2*x[2]))  # 최종 목적함수 = 판매이익 + 기계1 임대이익 + 기계3 임대이익 
solver.Maximize(obj)

print(f"Solving with {solver.SolverVersion()}")

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("총 이익 = %.1f (만원)" %(solver.Objective().Value())) 
    print("( 제품1 = 유모차 , 제품2 = 보행기 , 제품3 = 자전거 )")
    for j in range(data["num_vars"]):
        print('제품%d 구입량 = %.1f (대) '%(j+1 , x[j].solution_value()))
    print("기계1 대여시간 = %.1f (시간)" %(240-(8*x[0].solution_value() + 3*x[1].solution_value() + 3*x[2].solution_value())))
    print("기계3 대여시간 = %.1f (시간)" %(100-(4*x[0].solution_value() + 2*x[2].solution_value())))

else :
    print("The problem does not have an optimal solution.")

Solving with SCIP 9.2.2 [LP solver: SoPlex 7.1.3]
총 이익 = 1615.0 (만원)
( 제품1 = 유모차 , 제품2 = 보행기 , 제품3 = 자전거 )
제품1 구입량 = 0.0 (대) 
제품2 구입량 = 50.0 (대) 
제품3 구입량 = 0.0 (대) 
기계1 대여시간 = 90.0 (시간)
기계3 대여시간 = 100.0 (시간)
